Groundwater | Flow Modeling Track

# Step 8: Model Application — Using the Model to Answer Questions

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → **8-Apply** → 9-Document → 10-Communicate

**Story so far:** You have a calibrated MODFLOW 6 model of the Limmat Valley (Step 5). Cross-validation (Step 6) shows reasonable predictive power near the observation cluster, and FOSM sensitivity analysis (Step 7) highlights that predictions in the data-sparse western domain carry the highest uncertainty. Now it's time to use the model: the whole reason we built it.

| **Core content** | ~60-75 minutes |
|:--|:--|
| **Optional: Drought stress test** | +15-30 minutes |

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Design** a model scenario with a clear question, controlled changes, and appropriate outputs
2. **Implement** and run a new-pumping-well scenario using the calibrated model
3. **Compute** and interpret drawdown maps and water balance changes
4. **Delineate** well capture zones using MODFLOW 6 particle tracking (PRT)
5. **Relate** capture zones to Swiss groundwater protection zone regulations (S1, S2, S3)
6. **Compare** numerical capture zones with the analytical solution for a confined aquifer
7. *(optional)* Evaluate system response to reduced recharge (climate stress testing)

---

## Prerequisites

- **Completed [04f_model_implementation.ipynb](04f_model_implementation.ipynb)** (hard prereq) and [05f_calibration.ipynb](05f_calibration.ipynb) (or allow auto-download of the pre-computed calibration, ~0.5 MB).

In [ ]:
# Setup
import sys
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

from IPython.display import display, HTML

# Add support modules to path
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import (
    load_base_simulation, format_budget_summary,
    compare_water_balances, build_refined_gwf_model,
    build_prt_model, load_prt_results,
    delineate_protection_zones, run_scenario_prt,
)
from boundary_utils import assign_well_cells
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

import calibration_utils as cu
from setup_pest_calibration import (
    apply_calibrated_parameters,
    ensure_notebook4_model_exists,
    ensure_calibration_workspace,
)

import flopy

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

# --- 1. Define paths and create fresh application workspace ---
DATA_DIR = get_default_data_folder()
sim_name = 'limmat_valley'
nb4_workspace = os.path.join(DATA_DIR, 'notebook4_model')
calib_workspace = os.path.join(DATA_DIR, 'calibration')
app_workspace = os.path.join(DATA_DIR, 'application')

ensure_notebook4_model_exists(DATA_DIR)
pest_ws = ensure_calibration_workspace(DATA_DIR)   # returns .../calibration/pest_setup

# Always start fresh
if os.path.exists(app_workspace):
    shutil.rmtree(app_workspace)
# The calibration bundle contains pest_setup only; calibrated K is applied below.
# See DESIGN_DOCS/optional_pest_calibration_plan.md Phase 4.3.
shutil.copytree(nb4_workspace, app_workspace)
workspace = app_workspace
print(f"Application workspace: {workspace}")

# --- 2. Load simulation ---
sim = load_base_simulation(workspace)
gwf = sim.get_model(sim_name)
modelgrid = gwf.modelgrid
modelgrid.set_coord_info(crs="EPSG:2056")

disv = gwf.get_package('DISV')
idomain = disv.idomain.array
top = disv.top.array.flatten()
botm = disv.botm.array.flatten()

# Load boundary and river GIS data
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

river_gis_path = os.path.join(os.path.dirname(boundary_path), 'AV_Gewasser_-OGD.gpkg')
river_all = gpd.read_file(river_gis_path)
river_gdf = river_all[
    river_all['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
    & river_all.intersects(boundary_gdf.geometry.iloc[0])
]

# Cell centre coordinates
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters

print(f"Grid type: {modelgrid.grid_type}")
print(f"Active cells: {(idomain.flatten() > 0).sum()}")

# --- 3. Apply calibrated K field + Sihl leakance + NPF flags ---
cal_result = apply_calibrated_parameters(
    gwf, pest_ws, modelgrid, river_gdf, boundary_gdf,
    variogram_range=600.0, anisotropy_angle=-30.0, anisotropy_scaling=3.0,
)
k_field_cal = cal_result['k_field']

# --- 4. Run baseline ---
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("Baseline model run failed.")

head_baseline = gwf.output.head().get_data()
heads_bl = head_baseline.flatten() if head_baseline.ndim > 1 else head_baseline
print(f"Baseline head range: {heads_bl[idomain.flatten() > 0].min():.2f}"
      f" \u2013 {heads_bl[idomain.flatten() > 0].max():.2f} m a.s.l.")

# Save baseline specific discharge (undisturbed flow field \u2014 used for analytical overlay)
spdis_baseline = gwf.output.budget().get_data(text='DATA-SPDIS')[0]

# Baseline water balance
budget_baseline = format_budget_summary(gwf, sim)

---

## 1 — Before You Run: Scenario Design Principles

A model is a tool for answering questions. Before running any scenario, work through checklists given hereafter.

### 1.1 - Five Questions Before Running Any Scenario

| # | Question | Purpose |
|---|----------|---------|
| 1 | **What is the question?** | Define the prediction target (e.g., "maximum drawdown at location X") |
| 2 | **Can the model answer it?** | Match model physics/resolution to the question |
| 3 | **What are the controlled changes?** | Only modify what the scenario requires |
| 4 | **What is the baseline for comparison?** | Scenarios are meaningless without a reference |
| 5 | **How uncertain is the answer?** | Use NB7 insights to qualify results |

### 1.2 - Our Model's Capability Assessment

| Characteristic | Can do | Cannot do |
|----------------|--------|-----------|
| **Steady-state** | Long-term equilibrium drawdown | Response times, recovery |
| **Single layer** | Depth-averaged flow | Vertical gradients, multi-aquifer systems |
| **Calibrated K (pilot points)** | Spatially varying conductivity | Sub-grid heterogeneity |
| **River boundary (RIV)** | Head-dependent exchange | Overbank flooding, channel morphology |
| **No transport** | Flow paths, travel times (via PRT) | Contaminant concentrations, dispersion |

In [ ]:
# Checkpoint 1: Can our steady-state model predict transient drawdown?
create_multiple_choice('task08_checkpoint_1')


---

## 2 — New Pumping Well Impact

**Scenario:** A water utility proposes a new pumping well in the western part of the Limmat Valley — the data-sparse area identified in NB7 as having the highest parameter uncertainty. We'll assess the steady-state impact on heads and water balance.

**The five questions applied:**
1. *Question the model is expected to adress:* What is the maximum drawdown and where does the water come from?
2. *Can the model answer it?* Yes — steady-state drawdown and water balance are appropriate uses.
3. *Controlled changes:* Add one WEL entry; keep everything else identical to the baseline.
4. *Baseline:* The calibrated model we just ran.
5. *Uncertainty:* High in the western domain (NB7) — results are indicative, not precise.

### 2.1.- Baseline heads and water balance

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8))
heads_plot = np.where(idomain.flatten() > 0, heads_bl, np.nan)
quick_model_plot(modelgrid, heads_plot, boundary_gdf=boundary_gdf,
                 cmap='coolwarm', colorbar_label='Head (m a.s.l.)',
                 title='Baseline Head Distribution (calibrated model)', ax=ax)
fig.tight_layout()
plt.show()

print("\nBaseline Water Balance:")
display(budget_baseline)

### 2.2 - Implement the additionnal well, 

The proposed well is added, and the scenrio is run with it included in order to compute the steady state scenario result. We look at the results in term of drawdown at the new well location.

In [ ]:
# --- 1 : Add proposed well in the western domain ---
# Pick a location in the western third of the active domain (data-sparse area)
active_mask = idomain.flatten() > 0
active_x = xc[active_mask]
active_y = yc[active_mask]

# Western third of the domain
x_threshold = np.percentile(active_x, 33)
west_mask = active_x < x_threshold

# Use the centroid of the western zone as our well location
well_x = float(np.median(active_x[west_mask]))
well_y = float(np.median(active_y[west_mask]))

# Create a GeoDataFrame for the proposed well
pumping_rate = -500.0  # m³/d (extraction)
well_gdf = gpd.GeoDataFrame(
    {'rate': [pumping_rate], 'name': ['proposed_well']},
    geometry=[Point(well_x, well_y)],
    crs='EPSG:2056',
)

# Find the DISV cell using assign_well_cells
well_spd = assign_well_cells(well_gdf, modelgrid, rate_column='rate')
well_cell_id = well_spd[0][0]  # ((layer, cell2d), rate) → cellid tuple
well_cell_flat = well_cell_id[-1] if isinstance(well_cell_id, tuple) else well_cell_id

print(f"Proposed well location: ({well_x:.0f}, {well_y:.0f})")
print(f"DISV cell ID: {well_cell_flat}")
print(f"Pumping rate: {pumping_rate:.0f} m³/d")
print(f"Local K: {k_field_cal.flatten()[well_cell_flat]:.1f} m/d")
print(f"Aquifer thickness: {(top - botm)[well_cell_flat]:.1f} m")

# --- 2: add WEL package for the new well  ---
wel = gwf.get_package('WEL')
original_wel_spd = wel.stress_period_data.get_data(0).copy()  # save for restoration

# Convert existing recarray to list of tuples and append the new well
existing_wells = [(rec['cellid'], rec['q']) for rec in original_wel_spd]
new_spd = existing_wells + [((0, well_cell_flat), pumping_rate)]
wel.stress_period_data.set_data(new_spd, key=0)

# Run scenario
sim.write_simulation()
success_pump, _ = sim.run_simulation(silent=True)
if not success_pump:
    raise RuntimeError("Pumping scenario failed.")

head_pumping = gwf.output.head().get_data()
heads_pump = head_pumping.flatten() if head_pumping.ndim > 1 else head_pumping
print("Pumping scenario completed successfully.")

# --- 2.4 Compute and plot drawdown ---
drawdown = heads_bl - heads_pump  # positive = head declined
dd_active = drawdown[active_mask]
dd_plot = np.where(active_mask, drawdown, np.nan)

# Maximum drawdown at the well
dd_at_well = drawdown[well_cell_flat]
print(f"Maximum drawdown at well cell: {dd_at_well:.3f} m")
print(f"Drawdown range: {dd_active.min():.4f} – {dd_active.max():.4f} m")

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
vmax = max(abs(dd_active.min()), abs(dd_active.max()))
quick_model_plot(modelgrid, dd_plot, boundary_gdf=boundary_gdf,
                 cmap='RdBu_r', colorbar_label='Drawdown (m)',
                 title=f'Drawdown from New Well (Q = {pumping_rate:.0f} m³/d)',
                 vmin=-vmax, vmax=vmax, ax=ax)

# Mark the well
ax.scatter(well_x, well_y, s=85, c='black', marker='v',
           edgecolors='white', linewidth=1.5, zorder=6, label='New well')
ax.legend(loc='lower left')
fig.tight_layout()
plt.show()

In [ ]:
# Checkpoint 2: Maximum drawdown at the well cell (m)
check_task_with_solution('task08_checkpoint_2')

### 2.3 - Impact of the added well on the water balance

We look at the change in water balance for the different components of the aquifer in comparison with the baseline scenario, to better understand the impact of the new well on the water budget.



In [ ]:
# --- 2.5 Water balance comparison ---
budget_pumping = format_budget_summary(gwf, sim)
compare_water_balances(budget_baseline, budget_pumping,
                       label_a="Baseline", label_b="Pumping")

print(f"\n  Pumping rate:       {pumping_rate:+.0f} m³/d")
print(f"  The budget must close: extra outflow ≈ extra inflow from boundaries.")

In [ ]:
# Checkpoint 3: Where does the pumped water come from?
create_multiple_choice('task08_checkpoint_3')

### 2.4 - Linearity check

For a confined aquifer, the steady-state flow equation $\nabla \cdot (T \nabla h) + W = 0$ is linear in head because $T = Kb$ is independent of $h$. This means that doubling the pumping rate should exactly double the drawdown.

However, our model uses convertible cells (`icelltype=1`), so technically $T$ depends on the saturated thickness (which depends on $h$), making the equation mildly nonlinear. However, because our drawdown (~1–3 m) is small relative to the aquifer thickness (~20–40 m), the system behaves approximately linearly. 

We test the linearity by doubling the pumping rate of the new well and looking at the new drawdown in the next code cell.

In [ ]:
# --- 2.7 Double the pumping rate and compare ---
double_rate = 2 * pumping_rate  # -1000 m³/d

new_spd_double = existing_wells + [((0, well_cell_flat), double_rate)]
wel.stress_period_data.set_data(new_spd_double, key=0)

sim.write_simulation()
success_double, _ = sim.run_simulation(silent=True)

if success_double:
    head_double = gwf.output.head().get_data()
    heads_dbl = head_double.flatten() if head_double.ndim > 1 else head_double
    dd_double = heads_bl - heads_dbl
    dd_double_at_well = dd_double[well_cell_flat]

    print(f"Drawdown at Q = {pumping_rate:.0f} m³/d:   {dd_at_well:.4f} m")
    print(f"Drawdown at Q = {double_rate:.0f} m³/d:  {dd_double_at_well:.4f} m")
    print(f"Ratio: {dd_double_at_well / dd_at_well:.3f}  (expected: 2.0 for linear system)")
else:
    print("Double-pumping scenario failed.")

# Restore original WEL data
wel.stress_period_data.set_data(existing_wells, key=0)
print("\nWEL package restored to baseline for future use.")

In [ ]:
# Checkpoint 4: Why does drawdown scale linearly?
create_multiple_choice('task08_checkpoint_4')

### 2.5 - Further considerations and conclusion of the well case study

**What we learned:**
- The drawdown cone is local, it extends a few hundred metres from the well before being absorbed by the river boundary.
- The river responds by increasing leakage into the aquifer, supplying the pumped water.
- The drawdown magnitude depends on the local K (from pilot-point calibration), which has high uncertainty in the western domain (see NB7).

**Caveats:**
- This is equilibrium drawdown — the actual drawdown would develop over days to weeks.
- The model is not validated near the proposed well location.
- Results should be treated as indicative, not as design values.
- **Non-linearity sources**:  Two sources of mild nonlinearity exist. The unconfined formulation where $T = K(h - z_{bot})$, and the RIV boundary which switches between head-dependent leakage and a constant rate when heads drop below the riverbed. In practice, the ratio is close enough to 2.0 that the linear approximation holds well for scenario planning.

---

## 3 — Capture Zone and Swiss Groundwater Protection Zones

Now we move to the central application: **where does the water pumped by the well come from?** This defines the well's **capture zone** — the area of the aquifer that contributes water to the well. In Switzerland, capture zones are directly linked to regulatory protection zones.

### 3.1 - Swiss Groundwater Protection Zones Official Definition (Grundwasserschutzzonen, GSchV Art. 29)

| Zone | Name | Definition | Key Restrictions | How We Delineate |
|------|------|-----------|-----------------|------------------|
| **S1** | Fassungsbereich | Immediate well surrounds | No construction, no access | Fixed distance (~10 m) |
| **S2** | Engere Schutzzone | **10-day groundwater travel time** isochrone | No hazardous storage, agricultural restrictions | **Forward particle tracking** |
| **S3** | Weitere Schutzzone | Full capture zone (or defined portion) | Land-use restrictions | **Forward particle tracking** |

**S2 is the most important zone to compute** — it determines where contamination could reach the well within 10 days, too fast for emergency response.

### 3.2 - Grid Refinement for Protection Zone (S2) Resolution

The S2 zone typically extends only 20–30 m from the well. This is comparable to our base Voronoi cell size of 50 m, meaning that with one particle per cell, the coarse grid cannot resolve this critical zone. 

We therefore locally refine the grid around the well to ~10 m cells within 200 m, using the `refine_grid_locally()` utility from `disv_grid_utils`. Triangle handles the transition from fine to coarse cells automatically. This requires rebuilding the GWF model on the refined grid before running PRT. You will follow the same workflow for your case study.

### 3.3 - Implementation with MODFLOW 6 Particle Tracking (PRT)

We use the **PRT** (Particle Tracking) model built into MODFLOW 6. PRT reads the flow field from our GWF simulation and tracks particles through the velocity field — cell by cell, face by face — using the actual heterogeneous K field and boundary conditions.
We release one particle at the centre of every active cell and track forward. Particles that terminate at the pumping well define its capture zone. The travel time from each particle's release point to the well gives us the isochrone data for protection zones. This is known to be the forward tracking for capture zones technique.

| PRT Component | Purpose |
|---------------|---------|
| **DISV** | Same Voronoi grid as the GWF model |
| **MIP** | Porosity and tracking zones (well = zone 1, river = zone 2) |
| **PRP** | Particle release points (one per active cell) |
| **FMI** | Links to GWF head and budget output files |
| **EMS** | Explicit (non-iterative) solution for particle tracking |

In [ ]:
# --- 3.1 Build locally-refined grid and GWF model ---
ref_ws = os.path.join(workspace, 'refined')

ref_result = build_refined_gwf_model(
    gwf,
    boundary_gdf=boundary_gdf,
    river_gdf=river_gdf,
    refine_points=[(well_x, well_y)],
    head_array=heads_pump,
    workspace=ref_ws,
    well_data=[(well_x, well_y, pumping_rate)],
    sim_name='limmat_refined',
    baseline_head_array=heads_bl,
)

ref_sim = ref_result['sim']
ref_gwf = ref_result['gwf']
mg_ref = ref_result['modelgrid']
gridprops_ref = ref_result['gridprops']
ncpl_ref = ref_result['ncpl']
heads_ref = ref_result['heads']
well_cell_ref = ref_result['well_cells'][0]
ref_sim_name = 'limmat_refined'

xc_ref = mg_ref.xcellcenters
yc_ref = mg_ref.ycellcenters

# Drawdown on refined grid (for plotting in later cells)
heads_bl_ref = ref_result['baseline_heads']
dd_ref_plot = heads_bl_ref - heads_ref
heads_ref_plot = heads_ref
vmax_ref = np.max(np.abs(dd_ref_plot))

# --- 3.2 Set up MODFLOW 6 Particle Tracking (PRT) on refined grid ---
porosity = 0.15  # alluvial gravels

prt_result = build_prt_model(
    ref_gwf,
    porosity=porosity,
    well_cells=[well_cell_ref],
    tracking_time=365.0,
    sim_name='limmat_prt',
)
prt_sim = prt_result['prt_sim']
trackcsv_path = prt_result['trackcsv_path']

# --- 3.3 Run PRT and load results ---
success_prt, buff_prt = prt_sim.run_simulation()
if not success_prt:
    raise RuntimeError("PRT simulation failed — check listing file.")

ref_modelgrid = ref_gwf.modelgrid
prt_results = load_prt_results(trackcsv_path, ref_modelgrid)

prt_df = prt_results['prt_df']
terminal = prt_results['terminal']
captured_ids = prt_results['captured_ids']
captured_df = prt_results['captured_df']
travel_times = prt_results['travel_times']

# Starting positions of captured particles (their release cells = capture zone)
if len(captured_ids) > 0:
    start_records = captured_df.groupby('irpt').first()


# --- 3.4 Plot the capture zone (forward pathlines → well) ---
if len(captured_ids) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))

    # Base map: drawdown on refined grid (shows refined cell structure)
    quick_model_plot(mg_ref, dd_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='RdBu_r', colorbar_label='Drawdown (m)',
                     vmin=-vmax_ref, vmax=vmax_ref,
                     title='Capture Zone (PRT forward pathlines, 1 year)',
                     ax=ax)

    # Plot pathlines coloured by travel time
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='blue', linewidth=0.8, alpha=0.6)

    # Mark particle start positions (source locations)
    ax.scatter(start_records['x_world'], start_records['y_world'],
               s=15, c='blue', alpha=0.6, zorder=5,
               label='Particle origins (capture zone)')

    # Mark well
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='Pumping well')

    # Zoom to capture zone extent
    all_x = captured_df['x_world'].values
    all_y = captured_df['y_world'].values
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left')
    fig.tight_layout()
    plt.show()
else:
    print("No particles captured by the well — check model setup.")

### 3.4 - Analytical Comparison

For a confined aquifer with uniform flow, the capture zone of a pumping well has a classical analytical solution (Bear, 1979). The dividing streamline is the boundary between water captured by the well and water flowing past, and it satisfies the following analytic equation:

$$x = \frac{y}{\tan\!\left(\dfrac{2\pi\, T\, i\, y}{Q}\right)}$$

where $T = Kb$ is transmissivity, $i$ is the regional hydraulic gradient, and $Q$ is the pumping rate. Key properties associated to the the dividing streamline are:

- **Stagnation point** (downstream limit): $x_s = \dfrac{Q}{2\pi\, T\, i}$
- **Maximum half-width** (far upstream): $y_{\max} = \pm\dfrac{Q}{2\, T\, i}$

We can compute the local $T$ and $i$ from our model, and overlay the analytical solution on the PRT capture zone. Deviations show the effect of heterogeneity, boundaries, and non-uniform flow on the capture zone delineation (which are actually also the reasons why we need numerical models).

In [ ]:
# --- 3.5 Analytical capture zone: compute parameters ---
if len(captured_ids) > 0:
    # Local aquifer properties at the well
    k_local = k_field_cal.flatten()[well_cell_flat]
    b_local = float((top - botm)[well_cell_flat])
    T_local = k_local * b_local
    Q = abs(pumping_rate)

    # Regional flow direction from baseline specific discharge (undisturbed field)
    qx_bl = spdis_baseline['qx'][well_cell_flat]
    qy_bl = spdis_baseline['qy'][well_cell_flat]
    q_mag = np.sqrt(qx_bl**2 + qy_bl**2)

    # Hydraulic gradient: q = K * i  →  |i| = |q| / K
    i_local = q_mag / k_local
    flow_angle = np.arctan2(qy_bl, qx_bl)

    print(f"Local properties at well:")
    print(f"  K = {k_local:.1f} m/d,  b = {b_local:.1f} m,  T = {T_local:.0f} m²/d")
    print(f"  Baseline specific discharge: qx={qx_bl:.5f}, qy={qy_bl:.5f} m/d")
    print(f"  Hydraulic gradient |i| = {i_local:.5f}")
    print(f"  Flow direction: {np.degrees(flow_angle):.1f}° from east")

    # Analytical capture zone (Bear, 1979) — in flow-aligned coordinates
    x_stag = Q / (2 * np.pi * T_local * i_local)
    y_max = Q / (2 * T_local * i_local)
    print(f"\nAnalytical predictions:")
    print(f"  Stagnation point: {x_stag:.0f} m downstream")
    print(f"  Max capture width: ±{y_max:.0f} m")

    # Compute dividing streamline and rotate to model coordinates
    y_range = np.linspace(-0.95 * y_max, 0.95 * y_max, 500)
    y_range = y_range[y_range != 0]
    arg = 2 * np.pi * T_local * i_local * y_range / Q
    x_analytical = y_range / np.tan(arg)

    cos_f, sin_f = np.cos(flow_angle), np.sin(flow_angle)
    x_model = well_x + x_analytical * cos_f - y_range * sin_f
    y_model = well_y + x_analytical * sin_f + y_range * cos_f
else:
    print("No pathlines available for comparison.")

    
# --- 3.5b Analytical capture zone: plot overlay ---
if len(captured_ids) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(mg_ref, dd_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='RdBu_r', colorbar_label='Drawdown (m)',
                     vmin=-vmax_ref, vmax=vmax_ref,
                     title='Capture Zone: Numerical (PRT) vs Analytical',
                     ax=ax)

    # Numerical pathlines
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='blue', linewidth=0.6, alpha=0.4)

    # Analytical dividing streamline
    ax.plot(x_model, y_model, color='orange', linewidth=2.5, linestyle='--',
            label=f'Analytical (T={T_local:.0f}, i={i_local:.4f})', zorder=5)

    # Well marker
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='Pumping well')

    # Zoom to capture zone extent
    all_x = np.concatenate([captured_df['x_world'].values, x_model])
    all_y = np.concatenate([captured_df['y_world'].values, y_model])
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left', fontsize=9)
    fig.tight_layout()
    plt.show()

    print("\nThe analytical solution assumes uniform flow in a homogeneous confined aquifer.")
    print("Deviations from the numerical pathlines reflect heterogeneous K, non-uniform")
    print("gradients, and proximity to river boundaries — the reasons we need numerical models.")


# --- 3.6 Delineate the 10-day isochrone (S2 boundary) ---
if len(captured_ids) > 0:
    zones = delineate_protection_zones(
        captured_df, terminal, well_xy=(well_x, well_y),
        s2_days=10.0, s1_radius=10.0, s3_ratio=0.2,
    )
    s2_ids = zones['s2_ids']
    s2_envelope = zones['s2_envelope']
    s3_envelope = zones['s3_envelope']

    # Plot protection zones on refined grid (zoomed to capture zone)
    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(mg_ref, heads_ref_plot, boundary_gdf=boundary_gdf,
                     cmap='coolwarm', colorbar_label='Head (m a.s.l.)',
                     title='Swiss Groundwater Protection Zones',
                     ax=ax)

    # S3: concave envelope of full capture zone
    if s3_envelope is not None and not s3_envelope.is_empty:
        s3_x_env, s3_y_env = s3_envelope.exterior.xy
        ax.fill(s3_x_env, s3_y_env, facecolor='#2196F3', alpha=0.15,
                edgecolor='#2196F3', linewidth=2.0, zorder=4,
                label='S3: capture zone')

    # S3 pathlines (on top of envelope for detail)
    for pid in captured_ids:
        pl = captured_df[captured_df['irpt'] == pid]
        ax.plot(pl['x_world'].values, pl['y_world'].values,
                color='#2196F3', linewidth=0.5, alpha=0.3)

    # S2: convex envelope of 10-day pathlines
    if s2_envelope is not None and not s2_envelope.is_empty:
        s2_x_env, s2_y_env = s2_envelope.exterior.xy
        ax.fill(s2_x_env, s2_y_env, facecolor='#FF9800', alpha=0.35,
                edgecolor='#FF9800', linewidth=2.0, zorder=5,
                label='S2: 10-day isochrone')

    # S1: 10 m buffer around well
    s1_circle = plt.Circle((well_x, well_y), 10, fill=True,
                            facecolor='red', alpha=0.4,
                            edgecolor='red', linewidth=2.5,
                            label='S1: 10 m radius', zorder=6)
    ax.add_patch(s1_circle)

    # Well marker
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=7, label='Pumping well')

    # Zoom to capture zone extent
    all_x = captured_df['x_world'].values
    all_y = captured_df['y_world'].values
    pad = 200
    ax.set_xlim(min(all_x.min(), well_x) - pad, max(all_x.max(), well_x) + pad)
    ax.set_ylim(min(all_y.min(), well_y) - pad, max(all_y.max(), well_y) + pad)

    ax.legend(loc='lower left', fontsize=9)
    fig.tight_layout()
    plt.show()

    print(f"\nProtection zone summary:")
    print(f"  S1 (Fassungsbereich): 10 m radius — no construction, no activities")
    if s2_envelope is not None and not s2_envelope.is_empty:
        print(f"  S2 (Engere Schutzzone): 10-day travel time — "
              f"area {zones['s2_area']:.0f} m², max extent {zones['s2_max_extent']:.0f} m from well")
    else:
        print(f"  S2 (Engere Schutzzone): no particles reached well within 10 days")
    print(f"  S3 (Weitere Schutzzone): full capture zone — "
          f"area {zones['s3_area']:.0f} m², {len(captured_ids)} cells contribute")
else:
    print("No pathlines available.")

In [ ]:
# Checkpoint 5: Farm within the 10-day isochrone — which zone?
create_multiple_choice('task08_checkpoint_5')

### 3.5 - Non-Stationarity of Capture Zones

The capture zone we just computed is based on a single steady-state flow field, which is a snapshot of long-term average conditions.
In reality, capture zones are non-stationary. The analytical solution confirms this: $y_{\max} = Q/(2Ti)$: if, for instance, gradient $i$ decreases, capture width should increases as well.

| Condition | Effect on Capture Zone |
|-----------|----------------------|
| Summer drought (low recharge, low river stage) | **Expands** — lower gradients, well draws from larger area |
| Winter high-flow (high recharge, high river stage) | **Shrinks** — steeper gradients push water past the well faster.  |
| Parameter uncertainty (different plausible K fields) | **Ensemble of zones** — each K realisation gives a different boundary |
| Porosity uncertainty ($n_e$ = 0.15–0.25 for alluvial gravel) | **S2 shifts by ~60%** — travel time $\propto n_e/q$, so higher porosity → larger S2 |

**Regulatory implication**: Swiss guidelines recommend to define protection zones on the most conservative (largest) capture zone. This means that one should use the dry-season scenario as a default (lowest gradients → largest zone), or use Monte Carlo with K-field realisations from PEST++ IES to produce a probabilistic boundary. In addition, the data-sparse western domain has the highest K uncertainty → capture zone uncertainty is largest precisely where we placed the well (Refer to Notebook 7).

In [ ]:
# Checkpoint 6: Why not base zones on a single steady-state run?
create_multiple_choice('task08_checkpoint_6')

---

## 4 — Optional: Drought Stress Test under Climate Change Scenario 

**Scenario:** Climate projections suggest 20–30% recharge reduction in coming decades. What happens to aquifer levels and the capture zone?

**The five questions:**
1. *Question:* How much do heads decline, and how does the capture zone change?
2. *Can the model answer it?* Partially — steady-state gives the new equilibrium, not the transient path.
3. *Controlled changes:* Reduce recharge by 30%; keep pumping, river, and boundaries the same.
4. *Baseline:* The pumping-well scenario (so we see the combined effect).
5. *Uncertainty:* Recharge is already uncertain; 30% reduction is a plausible stress test.

### 4.1 - Run drought scenario and read impacts

In [ ]:
# --- 4.2 Run drought scenario: 30% recharge reduction ---
# Restore pumping well in WEL package
new_spd_pump = existing_wells + [((0, well_cell_flat), pumping_rate)]
wel.stress_period_data.set_data(new_spd_pump, key=0)

# Use run_calibration_trial with rch_multiplier (auto-restores after run)
head_drought = cu.run_calibration_trial(
    sim, sim_name,
    rch_multiplier=0.7,  # 30% reduction
)

if head_drought is not None:
    heads_dr = head_drought.flatten() if head_drought.ndim > 1 else head_drought
    print("Drought scenario completed.")
    print(f"Head range: {heads_dr[active_mask].min():.2f}"
          f" – {heads_dr[active_mask].max():.2f} m a.s.l.")
else:
    print("Drought scenario FAILED.")

# --- 4.3 Head decline map (drought + pumping vs baseline) ---
if head_drought is not None:
    decline = heads_bl - heads_dr  # positive = head dropped
    decline_plot = np.where(active_mask, decline, np.nan)
    decline_active = decline[active_mask]
    mean_decline = np.nanmean(decline_active)

    print(f"Mean head decline: {mean_decline:.3f} m")
    print(f"Max head decline:  {decline_active.max():.3f} m")
    print(f"Min head decline:  {decline_active.min():.3f} m")

    fig, ax = plt.subplots(figsize=(12, 8))
    quick_model_plot(modelgrid, decline_plot, boundary_gdf=boundary_gdf,
                     cmap='YlOrRd', colorbar_label='Head Decline (m)',
                     title='Head Decline: Drought (−30% recharge) + Pumping Well',
                     ax=ax)
    ax.scatter(well_x, well_y, s=85, c='black', marker='v',
               edgecolors='white', linewidth=1.5, zorder=6, label='New well')
    ax.legend(loc='lower left')
    fig.tight_layout()
    plt.show()


# --- 4.4 Water balance under drought ---
# Use return_budget=True to get budget in a single call (auto-restores params)
_, budget_drought = cu.run_calibration_trial(
    sim, sim_name,
    rch_multiplier=0.7,
    return_budget=True,
)

if budget_drought is not None:
    compare_water_balances(budget_baseline, budget_drought,
                           label_a="Baseline", label_b="Drought+Pumping")
print("\nRecharge restored to calibrated values.")

In [ ]:
# Checkpoint 7 (*Optional*): Mean head decline (m) across active domain
check_task_with_solution('task08_checkpoint_7')

### 4.2 - Stretch Exercise: Capture Zone Under Drought

In this part, we re-run the PRT particle tracking under drought conditions. This directly illustrates the non-stationarity discussed above: the gradients now being lower, we expect to obtain a larger capture zone.

In [ ]:
# --- 4.5 Re-run PRT under drought conditions (on refined grid) ---
if len(captured_ids) > 0:
    dr_results = run_scenario_prt(
        ref_gwf, ref_sim,
        porosity=porosity,
        well_cells=[well_cell_ref],
        rch_multiplier=0.7,
        tracking_time=365.0,
        workspace=os.path.join(ref_ws, 'prt_drought'),
        sim_name='limmat_prt_dr',
    )

    if dr_results is not None:
        cap_ids_dr = dr_results['captured_ids']
        cap_df_dr = dr_results['captured_df']

        print(f"\nDrought PRT: {len(cap_ids_dr)} particles captured "
              f"(vs {len(captured_ids)} under average conditions)")

        # Compute common zoom extent from both capture zones
        all_cap_x = np.concatenate([
            captured_df['x_world'].values, cap_df_dr['x_world'].values
        ])
        all_cap_y = np.concatenate([
            captured_df['y_world'].values, cap_df_dr['y_world'].values
        ])
        pad = 200
        xlim = (min(all_cap_x.min(), well_x) - pad,
                max(all_cap_x.max(), well_x) + pad)
        ylim = (min(all_cap_y.min(), well_y) - pad,
                max(all_cap_y.max(), well_y) + pad)

        # Compare capture zones on refined grid
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        for ax, cids, cdf, title in zip(
            axes,
            [captured_ids, cap_ids_dr],
            [captured_df, cap_df_dr],
            ['Average Conditions', 'Drought (-30% recharge)'],
        ):
            quick_model_plot(mg_ref, heads_ref_plot, boundary_gdf=boundary_gdf,
                             cmap='coolwarm', colorbar_label='Head (m)',
                             title=title, ax=ax)
            for pid in cids:
                pl = cdf[cdf['irpt'] == pid]
                ax.plot(pl['x_world'].values, pl['y_world'].values,
                        color='blue', linewidth=0.6, alpha=0.5)
            ax.scatter(well_x, well_y, s=85, c='black', marker='v',
                       edgecolors='white', linewidth=1.5, zorder=6)
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)

        fig.suptitle('Capture Zone Comparison: Average vs Drought',
                     fontsize=14, y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print("Drought scenario PRT failed.")
else:
    print("No base pathlines available — skipping drought comparison.")

In [ ]:
# Checkpoint 8 (*Optional*): How does the capture zone change under drought?
create_multiple_choice('task08_checkpoint_8')

## 5 — Restore model to baseline state

Before moving to the next notebook, we take care of restoring the model to its baseline state.

In [ ]:
# --- Final cleanup: restore model to baseline state ---
wel.stress_period_data.set_data(existing_wells, key=0)
sim.write_simulation()
print("Model restored to baseline state.")

---

## 6 — Synthesis: Pre-Application Checklist

Before applying the model to any new scenario, verify:

| Item | What we did | |
|------|------------|---|
| **Objective clearly defined** | "What is the drawdown and where does the water come from?" | |
| **Model capable of answering** | Steady-state drawdown and water balance — yes | |
| **Prediction within validated domain** | Western domain is NOT validated (NB6/NB7) — caveat documented | |
| **Scenarios physically plausible** | Q = 500 m³/d is realistic for a municipal well | |
| **Boundary conditions consistent** | River stage and recharge unchanged (except in drought test) | |
| **Uncertainty considered** | NB7 shows high K uncertainty in western domain | |
| **Caveats documented** | Steady-state only, unvalidated area, single K realisation | |
| **Results reproducible** | Fresh workspace copy each run, fixed random seeds | |

**What a professional application would add:**
- Ensemble of K-field realisations (PEST++ IES) for probabilistic capture zones
- Transient simulation for response times
- Sensitivity runs varying K at the well location
- Formal uncertainty quantification on protection zone boundaries

---

## Summary

**What we did:**
- Applied the **Five Questions** framework to design a pumping well scenario
- Added a well in the data-sparse western domain and computed **steady-state drawdown**
- Showed that the river provides the water (head-dependent boundary response)
- Verified **linearity** of the steady-state flow equation
- Used MODFLOW 6 **PRT** (forward particle tracking) to delineate the well's capture zone
- Compared the numerical capture zone with the **analytical solution** (Bear, 1979)
- Mapped capture zone boundaries onto **Swiss protection zones** (S1, S2, S3)
- *(optional)* Tested **drought impacts** on heads and capture zone extent

**Key findings:**
- Drawdown is modest (~1–3 m) due to the productive alluvial aquifer and river recharge
- The river boundary limits drawdown extent and supplies the pumped water
- The 10-day isochrone (S2) extends ~100–300 m from the well
- Under drought, the capture zone **expands** — protection zones should be based on conservative scenarios

---

## What You're Taking Forward

| From this notebook | Used in |
|-------------------|---------|
| Scenario design workflow (Five Questions) | Group case study — design your geothermal scenario |
| Drawdown computation and interpretation | Group case study — assess thermal impacts |
| MODFLOW 6 PRT particle tracking | Group case study — thermal breakthrough analysis |
| Protection zone concepts | Professional practice |
| Analytical vs numerical comparison | Validating your own model results |

---

## Next Steps

**Continue to:** [09f_documentation.ipynb](09f_documentation.ipynb) — where we learn to document the modelling process for reproducibility and communication.

**Group work:** Apply these same techniques to your geothermal well doublet scenario.

---

# References

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling* (2nd ed.). Academic Press.
- Bear, J. (1979). *Hydraulics of Groundwater*. McGraw-Hill. Chapter 8: Wells.
- Pollock, D.W. (2016). User guide for MODPATH Version 7. USGS Open-File Report 2016-1086.
- Swiss Federal Council (2018). *Gewässerschutzverordnung (GSchV)*, SR 814.201. Art. 29: Grundwasserschutzzonen.
- Hill, M.C., Tiedeman, C.R. (2007). *Effective Groundwater Model Calibration*. Wiley.
- Modflow 6 Particle Tracking (PRT): Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D. (2023). Particle tracking for MODFLOW 6. USGS Techniques and Methods 6-A78.